# Computer Networks Project 2
William Miller, Marc Pham

## Introduction

“This project simulates NAPALM-based network automation using Python classes to represent network devices. This allows us to demonstrate configuration management, state retrieval, and reporting without requiring physical or virtual routers.”

## Motivation

Maybe add the capability of pinging other routers to see if all the IP addresses are configured correctly.

## Methodology

In [1]:
%pip install napalm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
from napalm import get_network_driver
import time
import json
import os
from IPython.display import display

In [64]:
class driver:
    def __init__(self, hostname):
        self.hostname = hostname
        self.running_config = []
        self.candidate_config = []

        self.interfaces = {
            "eth0": {"is_up": True, "ip": None},
            "eth1": {"is_up": True, "ip": None},
        }

In [65]:
def load_merge_candidate(self, config):
    print(f"\n[{self.hostname}] load_merge_candidate()")
    self.candidate_config = config

driver.load_merge_candidate = load_merge_candidate

In [66]:
def compare_config(self):
    print(f"[{self.hostname}] compare_config()")

    diff = []

    for line in self.candidate_config:
        if line not in self.running_config:
            diff.append(f"+ {line}")

    for line in self.running_config:
        if line not in self.candidate_config:
            diff.append(f"- {line}")

    return "\n".join(diff)

driver.compare_config = compare_config

In [68]:
def commit_config(self):
    print(f"[{self.hostname}] commit_config()")

    self.running_config = self.candidate_config.copy()

    current_interface = None

    for line in self.candidate_config:
        parts = line.strip().split()

        if len(parts) == 0:
            continue

        if parts[0] == "interface":
            current_interface = parts[1]

            if current_interface not in self.interfaces:
                self.interfaces[current_interface] = {
                    "is_up": True,
                    "ip": None
                }

        elif len(parts) >= 3 and parts[0] == "ip" and parts[1] == "address":
            if current_interface:
                self.interfaces[current_interface]["ip"] = parts[2]

def discard_config(self):
    print(f"[{self.hostname}] discard_config()")
    self.candidate_config = []
    
driver.commit_config = commit_config
driver.discard_config = discard_config

In [69]:
def get_facts(self):
    return {
        "hostname": self.hostname,
        "vendor": "mock_napalm_driver",
        "config_lines": len(self.running_config),
    }

def get_interfaces(self):
    return self.interfaces

driver.get_facts = get_facts
driver.get_interfaces = get_interfaces

In [71]:
# ==============================
# CONFIG LOADER
# ==============================
def load_device_configs(filename):
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()  # fallback for notebooks/interactive mode

    file_path = os.path.join(base_dir, filename)

    with open(file_path, "r") as f:
        return json.load(f)

def build_config(device_data):
    config = []

    config.append(f"hostname {device_data['hostname']}")

    for intf, ip in device_data["interfaces"].items():
        config.append(f"interface {intf}")
        if ip:
            config.append(f"ip address {ip}")

    return config    

In [72]:
def provision_devices(devices, config_file):
    configs = load_device_configs(config_file)

    print("\n===== PROVISIONING DEVICES =====")

    for name, data in configs.items():
        device = devices[name]

        config_lines = build_config(data)

        device.load_merge_candidate(config_lines)

        diff = device.compare_config()

        print(f"\n[{name}] CONFIG DIFF:")
        print(diff if diff else "No changes")

        if diff:
            device.commit_config()
        else:
            device.discard_config()

In [73]:
def generate_report(devices, output_file="network_report.json"):

    report = {}
    for name, device in devices.items():
        report[name] = {
            "facts": device.get_facts(),
            "interfaces": device.get_interfaces()
        }

    with open(output_file, "w") as f:
        json.dump(report, f, indent=4)

## Short Demonstration

This topology represents a simple hybrid network design with a core point-to-point backbone and two access networks. The link between R1 and R2 uses the 10.0.0.0/30 subnet, which is typically used for router-to-router connections because it provides exactly two usable IP addresses, one for each router interface. In addition to the backbone, each router connects to a separate /24 access network: R1 connects to the 192.168.1.0/24 subnet and R2 connects to the 192.168.2.0/24 subnet. These larger subnets represent local LAN segments or edge networks where end devices such as PCs or servers would reside. Overall, this design demonstrates a basic enterprise-style architecture with a structured separation between the core inter-router link and the access layer networks.


In [62]:
print("\nTopology: R1 ----(10.0.0.0/30)---- R2")
print("          |                      |")
print("     192.168.1.0/24      192.168.2.0/24")

# Create devices
devices = {
    "R1": driver(hostname="R1"),
    "R2": driver(hostname="R2")
}

# Provision devices
provision_devices(devices, "networks_project2/device_config1.json")

print("\n===== NETWORK REPORT =====")

for name, device in devices.items():
    print(f"\n--- {name} ---")
    print("FACTS:")
    display(device.get_facts())

    print("\nINTERFACES:")
    display(device.get_interfaces())

# Generate report
generate_report(devices, "networks_project2/network_report1.json")


Topology: R1 ----(10.0.0.0/30)---- R2
          |                      |
     192.168.1.0/24      192.168.2.0/24

===== PROVISIONING PHASE =====

[R1] load_merge_candidate()
[R1] compare_config()

[R1] CONFIG DIFF:
+ hostname R1-Core
+ interface eth0
+ ip address 10.0.0.1/30
+ interface eth1
+ ip address 192.168.1.1/24
[R1] commit_config()
[R1] Interfaces after commit:
{'eth0': {'is_up': True, 'ip': '10.0.0.1/30'}, 'eth1': {'is_up': True, 'ip': '192.168.1.1/24'}}
[R1] Commit successful

[R2] load_merge_candidate()
[R2] compare_config()

[R2] CONFIG DIFF:
+ hostname R2-Edge
+ interface eth0
+ ip address 10.0.0.2/30
+ interface eth1
+ ip address 192.168.2.1/24
[R2] commit_config()
[R2] Interfaces after commit:
{'eth0': {'is_up': True, 'ip': '10.0.0.2/30'}, 'eth1': {'is_up': True, 'ip': '192.168.2.1/24'}}
[R2] Commit successful

===== NETWORK REPORT =====

--- R1 ---
FACTS:


{'hostname': 'R1', 'vendor': 'mock_napalm_driver', 'config_lines': 5}


INTERFACES:


{'eth0': {'is_up': True, 'ip': '10.0.0.1/30'},
 'eth1': {'is_up': True, 'ip': '192.168.1.1/24'}}


--- R2 ---
FACTS:


{'hostname': 'R2', 'vendor': 'mock_napalm_driver', 'config_lines': 5}


INTERFACES:


{'eth0': {'is_up': True, 'ip': '10.0.0.2/30'},
 'eth1': {'is_up': True, 'ip': '192.168.2.1/24'}}

The backbone network is divided into multiple /30 subnets, each representing a point-to-point link between two routers. Each /30 subnet provides four addresses: a network address, two usable IPs (assigned to the connected routers), and a broadcast address. For example, the...
- 10.0.0.0/30 subnet connects R1 and R2 using 10.0.0.1 and 10.0.0.2
- 10.0.0.4/30 subnet connectes R2 and R3 with usable addresses 10.0.0.5 and 10.0.0.6
- 10.0.0.8/30 subnet connects R3 and R4 with usable addresses 10.0.0.9 and 10.0.0.10
- 10.0.0.12/30 subnet connects R4 and R1 with usable addresses 10.0.0.13 and 10.0.0.14     

This design ensures that each router-to-router link is isolated in its own subnet, optimizing IP usage and reflecting standard practices for point-to-point backbone connections. For the access / edge layer, let's assign the IP addresses, like so:
* 192.168.1.0/24 subnet connects R1 and R5 using 192.168.1.1 and 192.168.1.2
* 192.168.2.0/24 subnet connects R1 and R6 using 192.168.2.1 and 192.168.2.2
* 192.168.3.0/24 subnet connects R2 and R7 using 192.168.3.1 and 192.168.3.2
* 192.168.4.0/24 subnet connects R2 and R8 using 192.168.4.1 and 192.168.4.2
* 192.168.5.0/24 subnet connects R3 and R9 using 192.168.5.1 and 192.168.5.2
* 192.168.6.0/24 subnet connects R3 and R10 using 192.168.6.1 and 192.168.6.2
* 192.168.7.0/24 subnet connects R4 and R11 using 192.168.7.1 and 192.168.7.2
* 192.168.8.0/24 subnet connects R4 and R12 using 192.168.8.1 and 192.168.8.2


In [63]:
print("\n================ TOPOLOGY ================\n")

print("          CORE RING (BACKBONE)")
print("        R1 ----------- R2")
print("        |               |")
print("        |               |")
print("        R4 ----------- R3\n")

print("          ACCESS / EDGE LAYER")
print("     R5  R6        R7  R8")
print("     |    |        |    |")
print("     R1   R1       R2   R2\n")

print("     R9  R10      R11  R12")
print("     |    |        |    |")
print("     R3   R3       R4   R4\n")

# Create devices
devices = {
    "R1": driver(hostname="R1"),
    "R2": driver(hostname="R2"),
    "R3": driver(hostname="R3"),
    "R4": driver(hostname="R4"),
    "R5": driver(hostname="R5"),
    "R6": driver(hostname="R6"),
    "R7": driver(hostname="R7"),
    "R8": driver(hostname="R8"),
    "R9": driver(hostname="R9"),
    "R10": driver(hostname="R10"),
    "R11": driver(hostname="R11"),
    "R12": driver(hostname="R12"),
}

# Provision devices
provision_devices(devices, "networks_project2/device_config2.json")
generate_report(devices)

# Show live device state (important for demo proof)
print("LIVE DEVICE STATE CHECK")

for name, device in devices.items():
    print(f"\n--- {name} ---")
    print("FACTS:")
    display(device.get_facts())

    print("\nINTERFACES:")
    display(device.get_interfaces())

# Generate report
generate_report(devices, "networks_project2/network_report2.json")


================ TOPOLOGY ================

          CORE RING (BACKBONE)
        R1 ----------- R2
        |               |
        |               |
        R4 ----------- R3

          ACCESS / EDGE LAYER
     R5  R6        R7  R8
     |    |        |    |
     R1   R1       R2   R2

     R9  R10      R11  R12
     |    |        |    |
     R3   R3       R4   R4


===== PROVISIONING PHASE =====

[R1] load_merge_candidate()
[R1] compare_config()

[R1] CONFIG DIFF:
+ hostname R1-Core
+ interface eth0
+ ip address 10.0.0.1/30
+ interface eth1
+ ip address 10.0.0.14/30
+ interface eth2
+ ip address 192.168.1.1/24
+ interface eth3
+ ip address 192.168.2.1/24
[R1] commit_config()
[R1] Interfaces after commit:
{'eth0': {'is_up': True, 'ip': '10.0.0.1/30'}, 'eth1': {'is_up': True, 'ip': '10.0.0.14/30'}, 'eth2': {'is_up': True, 'ip': '192.168.1.1/24'}, 'eth3': {'is_up': True, 'ip': '192.168.2.1/24'}}
[R1] Commit successful

[R2] load_merge_candidate()
[R2] compare_config()

[R2] CONFIG DI

{'hostname': 'R1', 'vendor': 'mock_napalm_driver', 'config_lines': 9}


INTERFACES:


{'eth0': {'is_up': True, 'ip': '10.0.0.1/30'},
 'eth1': {'is_up': True, 'ip': '10.0.0.14/30'},
 'eth2': {'is_up': True, 'ip': '192.168.1.1/24'},
 'eth3': {'is_up': True, 'ip': '192.168.2.1/24'}}


--- R2 ---
FACTS:


...

{'eth0': {'is_up': True, 'ip': '192.168.7.2/24'},
 'eth1': {'is_up': True, 'ip': None}}


--- R12 ---
FACTS:


{'hostname': 'R12', 'vendor': 'mock_napalm_driver', 'config_lines': 3}


INTERFACES:


{'eth0': {'is_up': True, 'ip': '192.168.8.2/24'},
 'eth1': {'is_up': True, 'ip': None}}

## Summary and Conclusion

## Distribution of Work